In [2]:
import pandas as pd
import numpy as np

# Verileri yukle
hava_df = pd.read_csv("/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/data/turkiye_hava_verisi.csv")
sehir_df = pd.read_csv("/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/data/sehir_listesi.csv")

print("=" * 40)
print("HAVA VERISI GENEL BILGI")
print("=" * 40)
print(f"Toplam satir: {len(hava_df)}")
print(f"Sutunlar: {hava_df.columns.tolist()}")
print(f"\nEksik deger kontrolu:")
print(hava_df.isnull().sum())
print(f"\nIlk 5 satir:")
print(hava_df.head())

HAVA VERISI GENEL BILGI
Toplam satir: 21920
Sutunlar: ['tarih', 'max_sicaklik', 'min_sicaklik', 'yagis', 'ruzgar_hizi', 'sehir', 'bolge']

Eksik deger kontrolu:
tarih           0
max_sicaklik    0
min_sicaklik    0
yagis           0
ruzgar_hizi     0
sehir           0
bolge           0
dtype: int64

Ilk 5 satir:
        tarih  max_sicaklik  min_sicaklik  yagis  ruzgar_hizi  sehir  \
0  2022-01-01          10.2          -0.8    0.0          5.2  Konya   
1  2022-01-02           8.4           2.1    1.5         17.3  Konya   
2  2022-01-03           7.8          -4.5    0.0          9.8  Konya   
3  2022-01-04          11.2          -0.7    0.0          9.1  Konya   
4  2022-01-05          12.5          -0.1    0.0         12.8  Konya   

        bolge  
0  Ic Anadolu  
1  Ic Anadolu  
2  Ic Anadolu  
3  Ic Anadolu  
4  Ic Anadolu  


In [3]:
# Bugday icin hava kosullarina gore etiket olustur
def bugday_etiketi(satir):
    max_s = satir["max_sicaklik"]
    min_s = satir["min_sicaklik"]
    yagis = satir["yagis"]
    ruzgar = satir["ruzgar_hizi"]

    # Uygun degil kosullari
    if min_s < -15:          # Asiri don riski
        return "uygun_degil"
    if max_s > 38:           # Asiri sicaklik
        return "uygun_degil"
    if yagis > 50:           # Asiri yagis, sel riski
        return "uygun_degil"
    if ruzgar > 60:          # Asiri ruzgar
        return "uygun_degil"

    # Riskli kosullar
    if min_s < -5:           # Don riski
        return "riskli"
    if max_s > 32:           # Yuksek sicaklik
        return "riskli"
    if yagis > 25:           # Fazla yagis
        return "riskli"
    if ruzgar > 40:          # Kuvvetli ruzgar
        return "riskli"

    # Uygun kosullar
    return "uygun"

# Etiketi uygula
hava_df["etiket"] = hava_df.apply(bugday_etiketi, axis=1)

print("Etiket dagilimi:")
print(hava_df["etiket"].value_counts())
print(f"\nYuzdelik dagilim:")
print(hava_df["etiket"].value_counts(normalize=True).round(2) * 100)

Etiket dagilimi:
etiket
uygun          17359
riskli          3885
uygun_degil      676
Name: count, dtype: int64

Yuzdelik dagilim:
etiket
uygun          79.0
riskli         18.0
uygun_degil     3.0
Name: proportion, dtype: float64


In [4]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Sayisal sutunlari normalize et
scaler = StandardScaler()
sayisal_sutunlar = ["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi"]

hava_df_normalized = hava_df.copy()
hava_df_normalized[sayisal_sutunlar] = scaler.fit_transform(hava_df[sayisal_sutunlar])

# Etiketi sayiya cevir
le = LabelEncoder()
hava_df_normalized["etiket_kod"] = le.fit_transform(hava_df["etiket"])

print("Etiket kodlari:")
for i, sinif in enumerate(le.classes_):
    print(f"  {i} → {sinif}")

print(f"\nNormalize edilmis ilk 5 satir:")
print(hava_df_normalized[sayisal_sutunlar + ["etiket", "etiket_kod"]].head())

Etiket kodlari:
  0 → riskli
  1 → uygun
  2 → uygun_degil

Normalize edilmis ilk 5 satir:
   max_sicaklik  min_sicaklik     yagis  ruzgar_hizi etiket  etiket_kod
0     -0.887149     -1.036406 -0.373763    -1.741860  uygun           1
1     -1.057262     -0.713490 -0.029617     0.312047  uygun           1
2     -1.113966     -1.448403 -0.373763    -0.961036  uygun           1
3     -0.792641     -1.025271 -0.373763    -1.079857  uygun           1
4     -0.669782     -0.958461 -0.373763    -0.451803  uygun           1


In [5]:
import joblib

DATA_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/data/"

# Islenmis veriyi kaydet
hava_df.to_csv(DATA_PATH + "islenmis_veri.csv", index=False)
hava_df_normalized.to_csv(DATA_PATH + "normalize_veri.csv", index=False)

# Scaler ve LabelEncoder'i kaydet (sonra modelda kullanacagiz)
joblib.dump(scaler, DATA_PATH + "../models/scaler.pkl")
joblib.dump(le, DATA_PATH + "../models/label_encoder.pkl")

print("Kaydedildi!")
print(f"  → data/islenmis_veri.csv       ({len(hava_df)} satir)")
print(f"  → data/normalize_veri.csv      ({len(hava_df_normalized)} satir)")
print(f"  → models/scaler.pkl")
print(f"  → models/label_encoder.pkl")

Kaydedildi!
  → data/islenmis_veri.csv       (21920 satir)
  → data/normalize_veri.csv      (21920 satir)
  → models/scaler.pkl
  → models/label_encoder.pkl
